# WebGIS Puskesmas Jawa Timur — environment preflight

This notebook checks only synthetic Point/MultiPolygon fixtures. It does not read or geocode project records from `Archives/`. Run it from the repository root locally, or upload/clone the repository in Colab.

In [ ]:
%pip install --require-hashes -r requirements.lock

In [ ]:
from pathlib import Path
import json

import folium
import geopandas as gpd
from shapely.geometry import Point, shape

FIXTURE_DIRECTORY = Path("tests/data/wf-055")
if not FIXTURE_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "Upload or clone the repository and run this notebook from its root; the synthetic fixture directory is missing."
    )

print({"geopandas": gpd.__version__, "folium": folium.__version__})

In [ ]:
def load_geometry(filename):
    feature = json.loads((FIXTURE_DIRECTORY / filename).read_text(encoding="utf-8"))
    assert feature["properties"]["synthetic"] is True
    return shape(feature["geometry"])

facility_point = load_geometry("facility-point.geojson")
district_boundary = load_geometry("district-multipolygon.geojson")
assert isinstance(facility_point, Point)
assert district_boundary.geom_type == "MultiPolygon"
assert district_boundary.is_valid

facilities = gpd.GeoDataFrame(
    [{"fixture_id": "SYNTH-FAC-001"}],
    geometry=[facility_point],
    crs="EPSG:4326",
)
assert facilities.geometry.within(district_boundary).tolist() == [True]
print("Synthetic spatial preflight passed.")

In [ ]:
latitude, longitude = facility_point.y, facility_point.x
preview = folium.Map(location=[latitude, longitude], zoom_start=10)
folium.Marker(
    [latitude, longitude],
    popup="SYNTH-FAC-001 — synthetic fixture",
).add_to(preview)
preview